In [1]:
import joblib
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor


def build_pipeline(numeric_features: list[str], categorical_features: list[str], xgb_params: dict) -> Pipeline:            
    """                                                                                                                    
    Builds the complete scikit-learn pipeline including imputation, encoding, and the XGBoost model.                       
    """                                                                                                                    
    numeric_pipeline = Pipeline([                                                                                          
        ("imputer", SimpleImputer(strategy="median")),                                                                     
    ])                                                                                                                     
                                                                                                                            
    categorical_pipeline = Pipeline([                                                                                      
        ("imputer", SimpleImputer(strategy="most_frequent")),                                                              
        ("encoder", OneHotEncoder(handle_unknown="ignore")),                                                               
    ])                                                                                                                     
                                                                                                                            
    preprocessor = ColumnTransformer([                                                                                     
        ("numeric", numeric_pipeline, numeric_features),                                                                   
        ("categorical", categorical_pipeline, categorical_features),                                                       
    ])                                                                                                                     
                                                                                                                            
    model = XGBRegressor(**xgb_params)                                                                                     
                                                                                                                            
    pipeline = Pipeline([                                                                                                  
        ("preprocessor", preprocessor),                                                                                    
        ("model", model),                                                                                                  
    ])                                                                                                                     
                                                                                                                            
    return pipeline                                                                                                        
                                                                                                                            


In [2]:
def evaluate_predictions(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:                                       
    """Calculates standard regression metrics."""                                                                          
    return {                                                                                                               
        "model": name,                                                                                                     
        "MAE": mean_absolute_error(y_true, y_pred),                                                                        
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,                                                                 
        "MedianAE": median_absolute_error(y_true, y_pred),                                                                 
        "R2": r2_score(y_true, y_pred),                                                                                    
    }                                                                                                                      
                                                                                                                            
def save_model_bundle(pipeline: Pipeline, metadata: dict, filepath: str):                                                  
    """Saves the trained pipeline and associated metadata to disk."""                                                      
    bundle = {"pipeline": pipeline, **metadata}                                                                            
    joblib.dump(bundle, filepath) 

In [3]:
                                                                                                                        
import numpy as np
import pandas as pd

from src.config import MODELS_DIR, RAW_DATA_PATH, XGB_PARAMS
from src.data import chronological_split, load_raw_data
from src.features import clean_bikes_data, engineer_time_series_features
from src.models import build_pipeline, evaluate_predictions, save_model_bundle


def prepare_data():
    print("1. Loading Data...")                                                                                            
    raw_df = load_raw_data(RAW_DATA_PATH)                                                                                  
                                                                                                                            
    print("2. Cleaning and Engineering Features...")                                                                       
    clean_df = clean_bikes_data(raw_df)                                                                                    
    features_df = engineer_time_series_features(clean_df, forecast_steps=1)                                                
                                                                                                                            
    print("3. Splitting Data (Chronological)...")                                                                          
    train_df, val_df, test_df = chronological_split(features_df)                                                           
                                                                                                                            
    # Combine train and validation for final training (like you did in notebook 3!)                                        
    train_val_df = pd.concat([train_df, val_df], ignore_index=True)                                                        
                                                                                                                            
    # Define features                                                                                                      
    numeric_features = [                                                                                                   
        "BIKE STANDS", "LATITUDE", "LONGITUDE", "AVAILABLE_BIKES_LAG_1",                                                   
        "AVAILABLE_BIKES_LAG_2", "AVAILABLE_BIKES_LAG_4", "AVAILABLE_BIKES_LAG_8",                                         
        "ROLLING_MEAN_4", "ROLLING_STD_4", "ROLLING_MEAN_8", "MINUTES_SINCE_PREVIOUS",                                     
        "HOUR_SIN", "HOUR_COS", "DOW_SIN", "DOW_COS", "IS_WEEKEND"                                                         
    ]                                                                                                                      
    categorical_features = ["STATION ID", "STATUS"]                                                                        
    target_column = "TARGET_AVAILABLE_BIKES"                                                                               

    X_train_val = train_val_df[numeric_features + categorical_features]                                                    
    y_train_val = train_val_df[target_column]                                                                              
    X_test = test_df[numeric_features + categorical_features]                                                              
    y_test = test_df[target_column]  

    return X_train_val, y_train_val, X_test, y_test, numeric_features, categorical_features, test_df

def train_XGB_regressor():
    X_train_val, y_train_val, X_test, y_test, numeric_features, categorical_features, test_df = prepare_data()
                                                                                      
                                                                                                                            
    print("1. Training Pipeline...")                                                                                       
    pipeline = build_pipeline(numeric_features, categorical_features, XGB_PARAMS)                                          
    pipeline.fit(X_train_val, y_train_val)                                                                                 
                                                                                                                            
    print("2. Evaluating on Test Set...")                                                                                  
    test_pred = pipeline.predict(X_test)                                                                                   
                                                                                                                            
    # Clip predictions to valid bike stands capacity                                                                       
    test_pred = np.clip(test_pred, 0, test_df["BIKE STANDS"].to_numpy())                                                   
                                                                                                                            
    metrics = evaluate_predictions("XGBoost", y_test, test_pred)                                                           
    print(f"Test Results: {metrics}")                                                                                      
                                                                                                                            
    print("3. Saving Model Bundle...")                                                                                     
    MODELS_DIR.mkdir(parents=True, exist_ok=True)                                                                          
    bundle_metadata = {                                                                                                    
        "feature_columns": numeric_features + categorical_features,                                                        
        "numeric_features": numeric_features,                                                                              
        "categorical_features": categorical_features,                                                                      
        "test_metrics": [metrics]                                                                                          
    }

def run_pipeline():                                                                                                        
    prepare_data()
    train_XGB_regressor()
    print("Pipeline Complete!")
 

ImportError: cannot import name 'build_pipeline' from 'src.models' (/mnt/WorkSpace/Repos/Dublin-Bikes/src/models.py)

In [ ]:
run_pipeline()

In [4]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from src.config import XGB_PARAMS, LGB_PARAMS

In [7]:
models_dict = {XGBRegressor: XGB_PARAMS,
            LGBMRegressor: LGB_PARAMS}

for model in models_dict:
    print(model.__name__)
    params = models_dict[model]
    print(params)

XGBRegressor
{'n_estimators': 500, 'learning_rate': 0.04, 'max_depth': 6, 'min_child_weight': 3, 'subsample': 0.85, 'colsample_bytree': 0.85, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}
LGBMRegressor
{'n_estimators': 500, 'learning_rate': 0.04, 'max_depth': 6, 'num_leaves': 31, 'min_child_weight': 3, 'subsample': 0.85, 'colsample_bytree': 0.85, 'objective': 'regression', 'random_state': 42, 'n_jobs': -1}


In [8]:
from src.models import build_preprocessor

